In [ ]:
import numpy as np
from subprocess import PIPE, run
import matplotlib.pyplot as plt
import os
import textwrap

class ExptBuilder():
    def __init__(self):
        self.__code_path__ = os.environ.get('code')
        self.__temp_exp_path__ = os.path.join(self.__code_path__, "k-exp", "kexp", "experiments", "slosh_expt.py")

    def run_expt(self):
        expt_path = self.__temp_exp_path__
        run_expt_command = r"%kpy% & ar " + expt_path
        result = run(run_expt_command, stdout=PIPE, stderr=PIPE, universal_newlines=True, shell=True)
        print(result.returncode, result.stdout, result.stderr)
        os.remove(self.__temp_exp_path__)
        return result.returncode
    
    def write_experiment_to_file(self, program):
        with open(self.__temp_exp_path__, 'w') as file:
            file.write(program)
    
    def expt(self, t_start, t_interval=50e-6, n_points=25):
        script = textwrap.dedent(f"""      
            import numpy as np
            from artiq.experiment import *
            from artiq.language.core import delay, kernel
            from kexp import Base, img_types, cameras


            class sloshy(EnvExperiment, Base):

                def prepare(self):
                    Base.__init__(self,setup_camera=True,save_data=True,
                                camera_select=cameras.andor,
                                imaging_type=img_types.ABSORPTION)
                    
                    self.xvar('t_tweezer_hold', np.linspace({t_start},{t_start+t_interval},{n_points}))

                    self.p.t_tof = 80.e-6
                    self.p.N_repeats = 10

                    self.finish_prepare(shuffle=True)

                @kernel
                def scan_kernel(self):

                    self.set_imaging_detuning(frequency_detuned=self.p.frequency_detuned_hf_f1m1)
                    self.imaging.set_power(self.camera_params.amp_imaging)

                    self.prepare_hf_tweezers(squeeze=True)
                    
                    delay(self.p.t_tweezer_hold)
                    self.tweezer.off()

                    delay(self.p.t_tof)
                    self.abs_image()

                @kernel
                def run(self):
                    self.init_kernel()
                    self.load_2D_mot(self.p.t_2D_mot_load_delay)
                    self.scan()

                def analyze(self):
                    import os
                    expt_filepath = os.path.abspath(__file__)
                    self.end(expt_filepath)
                """)
        return script

In [4]:
eBuilder = ExptBuilder()

In [ ]:
ts = np.linspace(0.,450e-6, 10)
for t in ts:
    print(f't_hold = {t*1.e6:1.2f} us')
    eBuilder.write_experiment_to_file(eBuilder.expt(t_start=t, n_points=10))
    eBuilder.run_expt()

t_hold = 0.00 us
0  100 values of t_tweezer_hold. 100 total shots. 300 total images expected.
Run ID: 67554
Acknowledged camera ready signal.
Camera is ready.

Sent: {'mask': 'spot', 'center': [994, 820], 'phase': 0.0, 'dimension': 0, 'initialize': False, 'spacing': 10, 'angle': 45}
-> mask: spot, dimension = 0 um, phase = 0.0 pi, x-center = 994, y-center = 820

 Run ID: 67554
shot 1/100 done
shot 2/100 done
shot 3/100 done
shot 4/100 done
shot 5/100 done
shot 6/100 done
shot 7/100 done
shot 8/100 done
shot 9/100 done
shot 10/100 done
shot 11/100 done
shot 12/100 done
shot 13/100 done
shot 14/100 done
shot 15/100 done
shot 16/100 done
shot 17/100 done
shot 18/100 done
shot 19/100 done
shot 20/100 done
shot 21/100 done
shot 22/100 done
shot 23/100 done
shot 24/100 done
shot 25/100 done
shot 26/100 done
shot 27/100 done
shot 28/100 done
shot 29/100 done
shot 30/100 done
shot 31/100 done
shot 32/100 done
shot 33/100 done
shot 34/100 done
shot 35/100 done
shot 36/100 done
shot 37/100 done
